# Industrial Surface Defect Detection Using Deep Learning and Explainable AI
## Milestone 2 — Custom CNN Baseline and Transfer Learning Smoke Tests

**Dataset:** MVTec AD — Tile  
**Primary task:** Binary classification (`good` vs. `defective`)  
**Input size:** 224 × 224  
**Fixed split source:** `tile_supervised_splits.csv`

This notebook establishes a reproducible modeling pipeline, runs a short Custom CNN smoke test, and defines a MobileNetV2 transfer-learning model. Full training, fine-tuning, threshold selection, and final test evaluation are intentionally deferred until the smoke tests and runtime plan are approved.

## 1. Experimental safeguards

- The leakage-safe manifests from Milestone 1 remain fixed.
- Augmentation is active only while training.
- Class weights are calculated from the full training split only.
- Validation data is used for model development.
- Test data is loaded only as metadata and is not evaluated in smoke mode.
- Smoke results are pipeline checks and must not be reported as final model performance.
- Full training and fine-tuning require explicit approval.

In [1]:
from __future__ import annotations

import json
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
RUN_MODE = "smoke"
RUN_TRANSFER_SMOKE = True

PROJECT_ROOT = Path.cwd().resolve()
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "tile" / "manifests" / "tile_supervised_splits.csv"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
MODEL_DIR = OUTPUT_ROOT / "models" / "milestone2"
HISTORY_DIR = OUTPUT_ROOT / "histories" / "milestone2"
FIGURE_DIR = OUTPUT_ROOT / "figures" / "milestone2"
KERAS_CACHE_DIR = OUTPUT_ROOT / "keras_cache"

os.environ["KERAS_HOME"] = str(KERAS_CACHE_DIR)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow version: {tf.__version__}")
print(f"Available GPUs: {tf.config.list_physical_devices('GPU')}")
print(f"Run mode: {RUN_MODE}")
print(f"Transfer smoke enabled: {RUN_TRANSFER_SMOKE}")

TensorFlow version: 2.21.0
Available GPUs: []
Run mode: smoke
Transfer smoke enabled: True


## 2. Runtime modes

| Mode | Images | Epochs | Purpose |
|---|---:|---:|---|
| Smoke | 64 train / 32 validation | 1 | Verify paths, tensors, training, and metrics |
| Fast | Full dataset | 5 | Quick development run |
| Standard | Full dataset | up to 20 | Default experiment with early stopping |
| Full | Full dataset | up to 35 | Carefully approved extended run |

This executed notebook uses **Smoke** mode. No hyperparameter search is performed.

In [2]:
MODE_CONFIG = {
    "smoke": {"train_limit": 64, "validation_limit": 32, "epochs": 1},
    "fast": {"train_limit": None, "validation_limit": None, "epochs": 5},
    "standard": {"train_limit": None, "validation_limit": None, "epochs": 20},
    "full": {"train_limit": None, "validation_limit": None, "epochs": 35},
}
config = MODE_CONFIG[RUN_MODE]
display(pd.DataFrame(MODE_CONFIG).T)

,train_limit,validation_limit,epochs
smoke,64.0,32.0,1.0
fast,NaN,NaN,5.0
standard,NaN,NaN,20.0
full,NaN,NaN,35.0


## 3. Load and verify the fixed manifests

The test rows are verified but not passed to a model in smoke mode.

In [3]:
manifest = pd.read_csv(MANIFEST_PATH)
required_columns = {
    "source_path", "relative_path", "binary_class", "binary_label",
    "defect_type", "split", "group_id", "sha256",
}
missing_columns = required_columns - set(manifest.columns)
if missing_columns:
    raise ValueError(f"Missing manifest columns: {sorted(missing_columns)}")

assert len(manifest) == 347
assert manifest["relative_path"].is_unique
assert manifest.groupby("group_id")["split"].nunique().max() == 1
assert manifest.groupby("sha256")["split"].nunique().max() == 1
assert manifest["source_path"].map(lambda value: Path(value).is_file()).all()

train_frame = manifest.loc[manifest["split"] == "train"].copy()
validation_frame = manifest.loc[manifest["split"] == "validation"].copy()
test_frame = manifest.loc[manifest["split"] == "test"].copy()

print(f"Train images: {len(train_frame)}")
print(f"Validation images: {len(validation_frame)}")
print(f"Reserved test images: {len(test_frame)}")
display(pd.crosstab(manifest["split"], manifest["binary_class"]))

Train images: 242
Validation images: 54
Reserved test images: 51


binary_class,defective,good
split,,
test,12,39
train,58,184
validation,14,40


## 4. Create deterministic smoke subsets

Sampling is stratified by the binary label and is used only to make the pipeline test inexpensive. Full runs use every image from the fixed train and validation manifests.

In [4]:
def stratified_limit(frame: pd.DataFrame, limit: int | None, seed: int) -> pd.DataFrame:
    if limit is None or limit >= len(frame):
        return frame.sample(frac=1, random_state=seed).reset_index(drop=True)

    fractions = frame["binary_label"].value_counts(normalize=True)
    selected_parts = []
    remaining = limit
    labels = sorted(fractions.index)
    for position, label in enumerate(labels):
        subset = frame.loc[frame["binary_label"] == label]
        if position == len(labels) - 1:
            count = remaining
        else:
            count = max(1, round(limit * fractions[label]))
            remaining -= count
        selected_parts.append(subset.sample(n=min(count, len(subset)), random_state=seed + int(label)))
    return pd.concat(selected_parts).sample(frac=1, random_state=seed).reset_index(drop=True)


active_train = stratified_limit(train_frame, config["train_limit"], SEED)
active_validation = stratified_limit(validation_frame, config["validation_limit"], SEED + 10)

print("Active training subset:")
display(active_train["binary_class"].value_counts().rename("count").to_frame())
print("Active validation subset:")
display(active_validation["binary_class"].value_counts().rename("count").to_frame())

Active training subset:


,count
binary_class,
good,49
defective,15


Active validation subset:


,count
binary_class,
good,24
defective,8


## 5. Build the input pipeline

Images are decoded as RGB, resized to 224 × 224, and batched. Augmentation is not part of the validation pipeline. It is implemented as training-aware Keras layers inside each model.

In [5]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_resize(path: tf.Tensor, label: tf.Tensor):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_png(image_bytes, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
    image = tf.cast(image, tf.float32)
    return image, tf.cast(label, tf.float32)


def make_dataset(frame: pd.DataFrame, training: bool) -> tf.data.Dataset:
    paths = frame["source_path"].astype(str).to_numpy()
    labels = frame["binary_label"].astype(np.float32).to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(decode_resize, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    return dataset.prefetch(AUTOTUNE)


train_dataset = make_dataset(active_train, training=True)
validation_dataset = make_dataset(active_validation, training=False)

sample_images, sample_labels = next(iter(train_dataset))
print(f"Image batch shape: {sample_images.shape}")
print(f"Label batch shape: {sample_labels.shape}")
print(f"Pixel range: {sample_images.numpy().min():.1f} to {sample_images.numpy().max():.1f}")

Image batch shape: (16, 224, 224, 3)
Label batch shape: (16,)
Pixel range: 19.3 to 255.0


## 6. Training-only augmentation and class weights

The augmentation policy is deliberately conservative for tile textures: small flips, rotations, zooms, contrast changes, and translations. No augmented image is written to disk.

Class weights use all 242 training images, not the validation or test splits.

In [6]:
augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
        tf.keras.layers.RandomRotation(0.05, fill_mode="reflect", seed=SEED),
        tf.keras.layers.RandomZoom(0.08, fill_mode="reflect", seed=SEED),
        tf.keras.layers.RandomTranslation(0.05, 0.05, fill_mode="reflect", seed=SEED),
        tf.keras.layers.RandomContrast(0.10, seed=SEED),
    ],
    name="training_augmentation",
)

train_counts = train_frame["binary_label"].value_counts().sort_index()
class_weights = {
    int(label): float(len(train_frame) / (len(train_counts) * count))
    for label, count in train_counts.items()
}
print("Full training counts:")
display(train_counts.rename("count").to_frame())
print("Candidate class weights:", class_weights)

Full training counts:


,count
binary_label,
0,184
1,58


Candidate class weights: {0: 0.657608695652174, 1: 2.086206896551724}


## 7. Custom CNN baseline

The baseline is intentionally compact and interpretable:

1. rescale pixels to [0, 1];
2. apply training-only augmentation;
3. learn three convolutional feature levels;
4. stabilize optimization with batch normalization;
5. reduce parameters with global average pooling;
6. regularize with dropout;
7. output a defective-class probability with a sigmoid.

The model uses binary cross-entropy and Adam. Recall and PR-AUC are included because missing a defective tile is operationally important.

In [7]:
def classification_metrics():
    return [
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
    ]


def build_custom_cnn() -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name="image")
    x = tf.keras.layers.Rescaling(1.0 / 255)(inputs)
    x = augmentation(x)
    for filters in (32, 64, 128):
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.35, seed=SEED)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="defective_probability")(x)
    model = tf.keras.Model(inputs, outputs, name="custom_cnn_baseline")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=classification_metrics(),
    )
    return model


baseline_model = build_custom_cnn()
baseline_model.summary()
print(f"Total parameters: {baseline_model.count_params():,}")

Model: "custom_cnn_baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ training_augmentation           │ (None, 224, 224, 3)    │             0 │
│ (Sequential)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ defective_probability (Dense)   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 94,049 (367.38 KB)

 Trainable params: 93,601 (365.63 KB)

 Non-trainable params: 448 (1.75 KB)

Total parameters: 94,049


## 8. Execute the Custom CNN smoke test

These results only verify that the complete pipeline trains and validates without error. One epoch on small subsets cannot establish model quality.

In [8]:
smoke_start = time.perf_counter()
baseline_smoke_history = baseline_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=config["epochs"],
    class_weight=class_weights,
    verbose=2,
)
baseline_smoke_seconds = time.perf_counter() - smoke_start

print(f"Baseline smoke runtime: {baseline_smoke_seconds:.1f} seconds")
print("Smoke test completed. Metrics above are not final performance estimates.")

4/4 - 37s - 9s/step - accuracy: 0.4688 - loss: 0.8009 - pr_auc: 0.2961 - precision: 0.2683 - recall: 0.7333 - roc_auc: 0.5714 - val_accuracy: 0.7500 - val_loss: 0.6712 - val_pr_auc: 0.3274 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.6458
Baseline smoke runtime: 38.3 seconds
Smoke test completed. Metrics above are not final performance estimates.


## 9. Define the MobileNetV2 transfer-learning model

The ImageNet-pretrained backbone is frozen for the first transfer-learning stage. MobileNetV2 preprocessing maps pixels to the range expected by the pretrained network. Only the classification head is trainable.

The transfer smoke test is disabled in this execution because pretrained weights are not cached and downloading them is a separate external action.

In [9]:
def build_mobilenetv2_transfer(weights: str = "imagenet") -> tuple[tf.keras.Model, tf.keras.Model]:
    backbone = tf.keras.applications.MobileNetV2(
        input_shape=(*IMAGE_SIZE, 3),
        include_top=False,
        weights=weights,
    )
    backbone.trainable = False

    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name="image")
    x = augmentation(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = backbone(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.30, seed=SEED)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="defective_probability")(x)
    model = tf.keras.Model(inputs, outputs, name="mobilenetv2_transfer")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=classification_metrics(),
    )
    return model, backbone


if RUN_TRANSFER_SMOKE:
    transfer_model, transfer_backbone = build_mobilenetv2_transfer(weights="imagenet")
    transfer_model.summary()
    transfer_start = time.perf_counter()
    transfer_smoke_history = transfer_model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=1,
        class_weight=class_weights,
        verbose=2,
    )
    transfer_smoke_seconds = time.perf_counter() - transfer_start
    print(f"Transfer smoke runtime: {transfer_smoke_seconds:.1f} seconds")
else:
    print("Transfer smoke test not executed. Set RUN_TRANSFER_SMOKE=True after approving the weight download.")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


Model: "mobilenetv2_transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ training_augmentation           │ (None, 224, 224, 3)    │             0 │
│ (Sequential)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ defective_probability (Dense)   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

4/4 - 59s - 15s/step - accuracy: 0.6875 - loss: 0.7644 - pr_auc: 0.2997 - precision: 0.3333 - recall: 0.3333 - roc_auc: 0.5810 - val_accuracy: 0.6875 - val_loss: 0.6305 - val_pr_auc: 0.4404 - val_precision: 0.3750 - val_recall: 0.3750 - val_roc_auc: 0.5260
Transfer smoke runtime: 61.3 seconds


## 10. Full-training callbacks — defined but not executed

Standard training will use:

- `EarlyStopping` to stop when validation loss no longer improves;
- `ModelCheckpoint` to preserve the best validation model;
- `ReduceLROnPlateau` to lower the learning rate when progress stalls;
- `TerminateOnNaN` to stop invalid training immediately.

Full training is intentionally not started in this notebook execution.

In [10]:
def make_callbacks(model_name: str):
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint_path = MODEL_DIR / f"{model_name}_best.keras"
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True
        ),
        tf.keras.callbacks.ModelCheckpoint(
            checkpoint_path, monitor="val_loss", save_best_only=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6
        ),
        tf.keras.callbacks.TerminateOnNaN(),
    ]


print("Callbacks are ready. No full training was executed.")

Callbacks are ready. No full training was executed.


## Conclusions

- The fixed leakage-safe manifests were loaded and revalidated.
- Training and validation input pipelines were created successfully.
- Augmentation is restricted to training behavior.
- Class weights were calculated from the training split only.
- The Custom CNN completed a one-epoch smoke test.
- MobileNetV2 transfer learning was defined in response to instructor feedback, but its pretrained weights and smoke test were not executed yet.
- The test split was not used.

### Next decision

Approve downloading the MobileNetV2 ImageNet weights and running its one-epoch smoke test. After both smoke tests pass, choose whether to begin the Standard Custom CNN and frozen-backbone MobileNetV2 training runs.